In [0]:
create table namaste_catalog.vineetdb.orders_op as 
select * 
from csv.`/Volumes/namaste_catalog/vineetdb/testvolume/orders/orders_10m.csv`
with ( header=True, inferSchema =True)

In [0]:
%python 
1024*1024*128

In [0]:
alter table namaste_catalog.vineetdb.orders_op set tblproperties ( "delta.targetFileSize" = "134217728")

In [0]:
OPTIMIZE namaste_catalog.vineetdb.orders_op

In [0]:
insert into namaste_catalog.vineetdb.orders_op
select * 
from csv.`/Volumes/namaste_catalog/vineetdb/testvolume/orders/orders_10m.csv`
with ( header=True, inferSchema =True)


### Now we would be enabign properties that we even while writing the data we would be making otpmization wit with out 10MB data we get 8 files of 13 MB each and now let us check

In [0]:
alter table namaste_catalog.vineetdb.orders_op set tblproperties (delta.autoOptimize.optimizeWrite = true)

In [0]:
insert into namaste_catalog.vineetdb.orders_op
select * 
from csv.`/Volumes/namaste_catalog/vineetdb/testvolume/orders/orders_10m.csv`
with ( header=True, inferSchema =True)


In [0]:
describe history namaste_catalog.vineetdb.orders_op

In [0]:
select _metadata.file_name , min(price) , max(price) , count(* ) 
from  namaste_catalog.vineetdb.orders_op
group by _metadata.file_name

In [0]:
optimize namaste_catalog.vineetdb.orders_op zorder by ( price) ;

In [0]:
describe history namaste_catalog.vineetdb.orders_op

In [0]:
select _metadata.file_name , min(price) , max(price) , count(*)
from namaste_catalog.vineetdb.orders_op
group by 1 

In [0]:
describe table namaste_catalog.vineetdb.orders_op

In [0]:
insert into  namaste_catalog.vineetdb.orders_op 
(select * 
from csv.`/Volumes/namaste_catalog/vineetdb/testvolume/orders/orders_20m.csv`
with ( header=True, inferSchema =True)
limit 100)


### Liquid Clustering

In [0]:
create table if not exists namaste_catalog.vineetdb.orders_cluster
as 
select * 
from csv.`/Volumes/namaste_catalog/vineetdb/testvolume/orders/orders_20m.csv`
with ( header=True, inferSchema =True)


In [0]:
alter table namaste_catalog.vineetdb.orders_cluster cluster by (price)

## Deep clone and shallow clone 
It generated the snapshot of the given version in time of the tables 
### Deep clone  metadata and data  both with fresh version
### Shallow clone only sotre metadata with fresh version

In [0]:
create table namaste_catalog.default.orders_sc shallow clone 
namaste_catalog.default.orders 

Deep clone is more superies than CTAS command it track version history till the snapshot taken from the source table

In [0]:
--drop table if exists namaste_catalog.default.orders

In [0]:
CREATE TABLE IF NOT EXISTS namaste_catalog.default.orders
USING DELTA
SELECT *
FROM CSV.`/Volumes/namaste_catalog/vineetdb/testvolume/orders/orders_10m.csv`
WITH ( HEADER TRUE, INFER_SCHEMA TRUE )
where order_id =200 

In [0]:
insert into namaste_catalog.default.orders 
(SELECT *
FROM CSV.`/Volumes/namaste_catalog/vineetdb/testvolume/orders/orders_10m.csv`
WITH ( HEADER TRUE, INFER_SCHEMA TRUE )
where order_id =2 )

In [0]:
select max(version)
from ( describe history namaste_catalog.default.orders ) 

In [0]:
create table namaste_catalog.default.orders_dc deep clone 
namaste_catalog.default.orders

In [0]:
describe history namaste_catalog.default.orders_dc

### Schema Drift 
- Solutions 1 from pyspark.sql.tyes import StructTypes 
- this is make sure the column expeced should be coming with expected name aloong with the data types 
- so right data goes to right column
- 
### Solution 2 merge into target t using source s 
- on t.id = s.id 
- when not matched then 
- insert * 
- and when matched then update set t.* =s.* 


### Type widening 
- two steps 
- 1. enable the properlty 
- 2.  now alter the table cs.t. alter column name type bigint
- 
- 

In [0]:
--/Volumes/namaste_catalog/vineetdb/testvolume/orders/
create table if nt